<a href="https://colab.research.google.com/github/Gowtham13042007/cron_job/blob/main/gae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf

tf.random.set_seed(1)

print('Num GPUs Available:',len(tf.config.list_physical_devices('GPU')))

Num GPUs Available: 1


In [ ]:
from tensorflow.keras.datasets import fashion_mnist

(x_train,y_train),(x_test,y_test) = fashion_mnist.load_data()

In [ ]:
import numpy as np

dataset=np.concatenate([x_train,x_test],axis=0)   #row wise joining
dataset=np.expand_dims(dataset,-1).astype('float32')/255 # makes to (70000,28,28,1)

In [ ]:
BATCH_SIZE=64

dataset=np.reshape(dataset,(-1,28,28,1))
dataset=tf.data.Dataset.from_tensor_slices(dataset)  # converts the numpy into tensorflow dataset objectd
dataset=dataset.shuffle(buffer_size=1024).batch(BATCH_SIZE)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

NOISE_DIM=150

generator=keras.models.Sequential(
    [
        keras.layers.InputLayer(input_shape=(NOISE_DIM,)),
        layers.Dense(7*7*256),
        layers.Reshape((7,7,256)),
        layers.Conv2DTranspose(256,3,activation=tf.nn.leaky_relu,strides=2,padding='same'),
        layers.Conv2DTranspose(128,3,activation=tf.nn.leaky_relu,strides=2,padding='same'),
        layers.Conv2DTranspose(1,3,activation='sigmoid',strides=1,padding='same'),
    ]
)

generator.summary()   # generator

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 12544)          │     1,894,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 14, 14, 256)    │       590,080 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 28, 28, 128)    │       295,040 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_2              │ (None, 28, 28, 1)      │         1,153 │
│ (Conv2DTranspose)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,780,417 (10.61 MB)

 Trainable params: 2,780,417 (10.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
discriminator=keras.models.Sequential([
    keras.layers.InputLayer(input_shape=(28,28,1)),
    layers.Conv2D(256,3,activation='relu',strides=2,padding='same'),
    layers.Conv2D(128,3,activation='relu',strides=2,padding='same'),
    layers.Flatten(),
    layers.Dense(64,activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1,activation='sigmoid')
])

discriminator.summary()  # discriminator

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 14, 14, 256)    │         2,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 7, 7, 128)      │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │       401,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 699,137 (2.67 MB)

 Trainable params: 699,137 (2.67 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
optimizerG=keras.optimizers.Adam(learning_rate=0.0001,beta_1=0.5)
optimizerD=keras.optimizers.Adam(learning_rate=0.0001,beta_1=0.5)
lossFn=keras.losses.BinaryCrossentropy()

gAccMetric=keras.metrics.BinaryAccuracy()
dAccMetric=keras.metrics.BinaryAccuracy()


In [ ]:
@tf.function
def trainDStep(data):
    batchsize = tf.shape(data)[0]
    noise = tf.random.normal((batchsize, NOISE_DIM))
    y_true = tf.concat(
        [
            tf.ones((batchsize,1)),
            tf.zeros((batchsize,1))
        ],
        axis=0
    )
    with tf.GradientTape() as tape:
        fake = generator(noise, training=True)
        x = tf.concat([data, fake], axis=0)
        y_pred = discriminator(x, training=True)
        discriminatorLoss = lossFn(y_true, y_pred)

    grads = tape.gradient(
        discriminatorLoss,
        discriminator.trainable_variables
    )

    optimizerD.apply_gradients(
        zip(grads, discriminator.trainable_variables)
    )

    dAccMetric.update_state(y_true, y_pred)

    return {
        'discriminator_loss': discriminatorLoss,
        'discriminator_accuracy': dAccMetric.result()
    }

In [ ]:
@tf.function
def trainGStep(data):
  batchsize=tf.shape(data)[0]
  noise=tf.random.normal(shape=(batchsize,NOISE_DIM))
  y_true=tf.ones(batchsize,1)

  with tf.GradientTape() as tape:
   y_pred=discriminator(generator(noise))
   generatorLoss=lossFn(y_true,y_pred)

  grads=tape.gradient(generatorLoss,generator.trainable_weights)
  optimizerG.apply_gradients(zip(grads,generator.trainable_weights))
  gAccMetric.update_state(y_true,y_pred)

  return {
      'generator_loss':generatorLoss,
      'generator_accuracy':gAccMetric.result()
  }

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

def plotImages(model):

    noise = np.random.normal(size=(81, NOISE_DIM))

    images = model(noise, training=False).numpy()

    plt.figure(figsize=(9,9))

    for i in range(81):

        plt.subplot(9,9,i+1)

        plt.imshow(images[i][:,:,0], cmap='gray')

        plt.axis('off')

    plt.tight_layout()

    plt.show(block=True)

In [ ]:
for epoch in range(30):

    dLossSum = 0
    gLossSum = 0
    dAccSum = 0
    gAccSum = 0
    cnt = 0

    for batch in dataset:

        dLoss = trainDStep(batch)
        gLoss = trainGStep(batch)

        dLossSum += dLoss['discriminator_loss']
        gLossSum += gLoss['generator_loss']

        dAccSum += dLoss['discriminator_accuracy']
        gAccSum += gLoss['generator_accuracy']

        cnt += 1

    print(
        'Epoch:', epoch,
        'DLoss:', dLossSum/cnt,
        'GLoss:', gLossSum/cnt
    )

    plotImages(generator)